# B2-020-language-transformers — Practice p19 — Solution

**Type:** scenario · **Difficulty:** core · **Concepts:** masked-language-modeling, nlp-pretraining-objectives, transformer-nlp-task-design

*65 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260812`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C11-neural-training`, `B2-019-attention-transformers`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb).

## Pinned held-out certificate

Use these literal held-out IDs: causal `[[4, 4, 4, 4, 4, 4, 0, 0], [5, 5, 5, 5, 5, 5, 0, 0]]`; MLM inputs `[[4, 1, 4, 4, 4, 4, 0, 0], [5, 5, 5, 5, 5, 1, 0, 0]]`; MLM labels `[[-100, 4, -100, -100, -100, -100, -100, -100], [-100, -100, -100, -100, -100, 5, -100, -100]]`.  Recompute held-out losses and require causal >= 1.020881 and mlm >= 0.969689 improvement over the independently reconstructed seeded-initial baseline.  Expected top-1 probes are `causal_row0_after_red=4`, `causal_row1_after_blue=5`, `mlm_row0_position1=4`, and `mlm_row1_position4=4`; compare losses and tensor-derived values with `ATOL=1e-5`, `RTOL=1e-5`.  These literals are evaluation assertions, not permission to load either generated state artifact.

## Solution

The protocol uses one optimizer continuously across both objectives. For a stated left-to-right generator, I would select the causal next-token objective because deployment cannot use right context; MLM remains useful encoder pretraining but mismatches that visibility contract.

In [ ]:
import importlib.util

def load_literal_module(name, relative_path):
    spec = importlib.util.spec_from_file_location(name, relative_path)
    assert spec is not None and spec.loader is not None
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)

class TinyEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(12, 8, padding_idx=0)
        self.position_embedding = nn.Embedding(8, 8)
        self.norm1 = nn.LayerNorm(8, eps=1e-5)
        self.attention = nn.MultiheadAttention(8, 2, dropout=0.0, batch_first=True)
        self.norm2 = nn.LayerNorm(8, eps=1e-5)
        self.ff1 = nn.Linear(8, 16)
        self.ff2 = nn.Linear(16, 8)
        self.mask_history = []

    def forward(self, token_ids, *, mask_mode):
        self.mask_history.append(mask_mode)
        length = token_ids.shape[1]
        positions = torch.arange(length, device=token_ids.device)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)
        normalized = self.norm1(x)
        attention_mask = None
        if mask_mode == "causal":
            attention_mask = torch.triu(
                torch.ones(length, length, dtype=torch.bool, device=token_ids.device),
                diagonal=1,
            )
        elif mask_mode != "bidirectional":
            raise ValueError(f"unknown mask mode: {mask_mode}")
        attended, _ = self.attention(
            normalized,
            normalized,
            normalized,
            attn_mask=attention_mask,
            key_padding_mask=token_ids.eq(0),
            need_weights=False,
        )
        x = x + attended
        return x + self.ff2(F.gelu(self.ff1(self.norm2(x))))

fixture = load_literal_module("language_fixture_p19", "../data/language_fixture.py")

def build_seeded_models():
    torch.manual_seed(20260812)
    return TinyEncoder(), nn.Linear(8, 12)

def heldout_metrics(encoder, head):
    causal_rows = torch.tensor([[4,4,4,4,4,4,0,0], [5,5,5,5,5,5,0,0]], dtype=torch.int64)
    causal_inputs, causal_targets = causal_rows[:, :-1], causal_rows[:, 1:]
    causal_logits = head(encoder(causal_inputs, mask_mode="causal"))
    valid = causal_targets.ne(0)
    causal_loss = F.cross_entropy(causal_logits[valid], causal_targets[valid], reduction="mean")
    mlm_inputs = torch.tensor([[4,1,4,4,4,4,0,0], [5,5,5,5,5,1,0,0]], dtype=torch.int64)
    mlm_labels = torch.tensor([[-100,4,-100,-100,-100,-100,-100,-100], [-100,-100,-100,-100,-100,5,-100,-100]], dtype=torch.int64)
    mlm_logits = head(encoder(mlm_inputs, mask_mode="bidirectional"))
    mlm_loss = F.cross_entropy(mlm_logits.reshape(-1, 12), mlm_labels.reshape(-1), ignore_index=-100, reduction="mean")
    probes = {
        "causal_row0_after_red": causal_logits.argmax(-1)[0, 0].item(),
        "causal_row1_after_blue": causal_logits.argmax(-1)[1, 0].item(),
        "mlm_row0_position1": mlm_logits.argmax(-1)[0, 1].item(),
        "mlm_row1_position4": mlm_logits.argmax(-1)[1, 4].item(),
    }
    return {"causal": causal_loss, "mlm": mlm_loss}, probes

def run_pretraining_protocol(fixture):
    encoder, head = build_seeded_models()
    optimizer = torch.optim.AdamW(
        [*encoder.parameters(), *head.parameters()], lr=0.03, weight_decay=0,
        betas=(0.9, 0.999), eps=1e-8, amsgrad=False, foreach=False, fused=False,
    )
    trace = []
    causal_rows = torch.tensor(fixture.CAUSAL_TRAIN_IDS, dtype=torch.int64)
    causal_inputs, causal_targets = causal_rows[:, :-1], causal_rows[:, 1:]
    causal_valid = causal_targets.ne(0)
    mlm_inputs = torch.tensor(fixture.MLM_INPUT_IDS, dtype=torch.int64)
    mlm_labels = torch.tensor(fixture.MLM_LABEL_IDS, dtype=torch.int64)
    first_parameter = next(iter(encoder.parameters()))
    for phase, count in (("causal", 40), ("mlm", 40)):
        for update_index in range(1, count + 1):
            optimizer.zero_grad(set_to_none=True)
            if phase == "causal":
                logits = head(encoder(causal_inputs, mask_mode="causal"))
                loss = F.cross_entropy(logits[causal_valid], causal_targets[causal_valid], reduction="mean")
                mask_mode = "causal"
            else:
                logits = head(encoder(mlm_inputs, mask_mode="bidirectional"))
                loss = F.cross_entropy(logits.reshape(-1,12), mlm_labels.reshape(-1), ignore_index=-100, reduction="mean")
                mask_mode = "bidirectional"
            loss_value = loss.item()
            loss.backward()
            optimizer.step()
            optimizer_step = int(optimizer.state[first_parameter]["step"].item())
            trace.append({
                "phase": phase, "update_index": update_index,
                "mask_mode": mask_mode, "optimizer_step": optimizer_step,
                "loss": loss_value,
            })
    encoder.pretraining_mask_history = tuple(encoder.mask_history)
    return encoder, head, trace

initial_encoder, initial_head = build_seeded_models()
initial_losses, _ = heldout_metrics(initial_encoder, initial_head)
encoder, head, phase_trace = run_pretraining_protocol(fixture)
final_losses, probes = heldout_metrics(encoder, head)
causal_trace = phase_trace[:40]
mlm_trace = phase_trace[40:]
REPORT = {
    "causal_phase_initial": causal_trace[0]["loss"],
    "causal_phase_final": causal_trace[-1]["loss"],
    "mlm_phase_initial": mlm_trace[0]["loss"],
    "mlm_phase_final": mlm_trace[-1]["loss"],
    "deployment_objective": "causal next-token modeling",
}

### Answer check

In [ ]:
assert len(phase_trace) == 80
assert [row["phase"] for row in phase_trace] == ["causal"] * 40 + ["mlm"] * 40
assert [row["update_index"] for row in causal_trace] == list(range(1, 41))
assert [row["update_index"] for row in mlm_trace] == list(range(1, 41))
assert [row["mask_mode"] for row in phase_trace] == ["causal"] * 40 + ["bidirectional"] * 40
assert list(encoder.pretraining_mask_history) == ["causal"] * 40 + ["bidirectional"] * 40
assert [row["optimizer_step"] for row in phase_trace] == list(range(1, 81))
assert causal_trace[0]["loss"] > causal_trace[-1]["loss"]
assert mlm_trace[0]["loss"] > mlm_trace[-1]["loss"]
assert initial_losses["causal"].item() - final_losses["causal"].item() >= 1.020881
assert initial_losses["mlm"].item() - final_losses["mlm"].item() >= 0.969689
assert probes == {"causal_row0_after_red": 4, "causal_row1_after_blue": 5, "mlm_row0_position1": 4, "mlm_row1_position4": 4}
assert REPORT["deployment_objective"] == "causal next-token modeling"